## Ingestion into AWS S3 storage using `dlt` and `boto3`.

In [132]:
import dlt, boto3, os, configparser, tarfile, gdown
import polars as pl
from pathlib import Path
import xml.etree.ElementTree as ET
os.makedirs('data', exist_ok=True)
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/xmls", exist_ok=True)
os.makedirs("data/parquet", exist_ok=True)

In [133]:
config = configparser.ConfigParser()
config.read(os.path.expanduser("~/.aws/config"))
bucket_name = config["profile hiv-project"]["s3_bucket"]

In [4]:
@dlt.resource(name="tce_xmls", write_disposition="replace")
def ingest_tce_xmls(
    gdrive_url: str = "https://drive.google.com/file/d/19PRBBOEFZGalIXE8am64L1e6dyBlF5K_/view?usp=sharing",
    tar_path: str = "data/raw/tce_xmls.tar.gz",
    extract_path: str = "data/xmls",
):
    os.makedirs(os.path.dirname(tar_path), exist_ok=True)
    gdown.download(gdrive_url, tar_path, quiet=False)

    with tarfile.open(tar_path, "r:*") as tar:
        tar.extractall(path=extract_path, filter="tar")

    for root, _, files in os.walk(extract_path):
        for fname in files:
            yield {"path": os.path.join(root, fname), "filename": fname}


pipeline = dlt.pipeline(
    pipeline_name="tce_ingestion",
    destination="filesystem",
    dataset_name="tce_raw",
)

load_info = pipeline.run(ingest_tce_xmls())
print(load_info)

Downloading...
From: https://drive.google.com/uc?id=19PRBBOEFZGalIXE8am64L1e6dyBlF5K_
To: /Users/benzenesea/Desktop/hiv-resistance-analytics/ingestion/data/raw/tce_xmls.tar.gz
100%|██████████| 24.8M/24.8M [00:03<00:00, 7.00MB/s]


Pipeline tce_ingestion load step completed in 1.51 seconds
1 load package(s) were loaded to destination filesystem and into dataset tce_raw
The filesystem destination used s3://hiv-data-022784797781 location to store data
Load package 1776550242.910173 is LOADED and contains no failed jobs


---

Parse the XML files into Parquet files.

In [ ]:
def parse_all(xml_path):
    root = ET.parse(xml_path).getroot()

    filename = Path(xml_path).name
    patient = root.findtext('patient/patientAlias')
    baseline_year = root.findtext('baselineYear')
    cd4_nadir = root.findtext('patient/CD4NadirBeforeTCE')
    date_unit = root.findtext('dateUnit')
    schema_version = root.findtext('schemaVersion')

    case_rows = [{
        "xml_filename": filename,
        "patient_alias": patient,
        "baseline_year": baseline_year,
        "cd4_nadir_before_tce": cd4_nadir,
        "date_unit": date_unit,
        "schema_version": schema_version
    }]

    measurements = []
    isolates = []
    mutations = []
    treatments = []

    # measurements
    for tag, mtype, val_fields in [
        ("baselineRNA", "RNA", ["logLoad", "rawLoad"]),
        ("pastRNA", "RNA", ["logLoad", "rawLoad"]),
        ("followupRNA", "RNA", ["logLoad", "rawLoad"]),
        ("baselineCD4", "CD4", ["count"]),
        ("pastCD4", "CD4", ["count"]),
        ("followupCD4", "CD4", ["count"]),
    ]:
        for node in root.findall(f".//{tag}"):
            for vf in val_fields:
                val = node.findtext(vf)
                if val:
                    measurements.append({
                        "xml_filename": filename,
                        "patient_alias": patient,
                        "relative_date": node.findtext("relativeDate"),
                        "measurement_type": mtype,
                        "timepoint_tag": tag,
                        "timepoint_type": tag.replace(mtype, "").lower(),
                        "value": val,
                        "value_type": vf
                    })

    # isolates + mutations
    for tag, itype in [("baselineIsolate", "baseline"), ("pastIsolate", "past")]:
        for i, iso in enumerate(root.findall(f".//{tag}")):
            iso_id = f"{filename}_{itype}_{i}"

            isolates.append({
                "xml_filename": filename,
                "patient_alias": patient,
                "isolate_id": iso_id,
                "isolate_type": itype,
                "gene": iso.findtext("gene"),
                "subtype": iso.findtext("subtype"),
                "relative_date": iso.findtext("relativeDate"),
                "aa_start": iso.findtext("aaStart"),
                "aa_stop": iso.findtext("aaStop"),
                "aa_sequence": iso.findtext("aaSequence"),
                "nucleotide_sequence": iso.findtext("nucleotideSequence"),
            })

            for mut in iso.findall(".//aaMutation"):
                mutations.append({
                    "xml_filename": filename,
                    "patient_alias": patient,
                    "isolate_id": iso_id,
                    "isolate_type": itype,
                    "gene": iso.findtext("gene"),
                    "relative_date": iso.findtext("relativeDate"),
                    "position": mut.findtext("position"),
                    "amino_acid": mut.findtext("aminoAcid"),
                    "mixtures": mut.findtext("mixtures"),
                })

    # treatments — NEW (baseline)
    for tnode in root.findall(".//baselineNewTreatment"):
        start = tnode.findtext("relativeStartDate")
        stop = tnode.findtext("relativeStopDate")
        duration = tnode.findtext("duration")
        for drug in tnode.findall("drug"):
            treatments.append({
                "xml_filename": filename,
                "patient_alias": patient,
                "relative_start_date": start,
                "relative_stop_date": stop,
                "duration": duration,
                "drug_code": drug.findtext("drugCode"),
                "drug_class": drug.findtext("drugClass"),
                "treatment_type": "new"
            })

    # treatments — ADD THIS (past)
    for tnode in root.findall(".//pastRegimenTreatments"):
        start = tnode.findtext("relativeStartDate")
        stop = tnode.findtext("relativeStopDate")
        duration = tnode.findtext("duration")
        for drug in tnode.findall("drug"):
            treatments.append({
                "xml_filename": filename,
                "patient_alias": patient,
                "relative_start_date": start,
                "relative_stop_date": stop,
                "duration": duration,
                "drug_code": drug.findtext("drugCode"),
                "drug_class": drug.findtext("drugClass"),
                "treatment_type": "past"
            })

    return case_rows, measurements, isolates, mutations, treatments

In [7]:
@dlt.resource(name="cases", write_disposition="replace")
def tce_cases(xml_dir="data/xmls/xmls_db"):
    for f in Path(xml_dir).glob("*.xml"):
        try:
            case_rows, _, _, _, _ = parse_all(f)
            yield from case_rows
        except Exception as e:
            print(f"Failed {f.name}: {e}")

@dlt.resource(name="measurements", write_disposition="replace")
def tce_measurements(xml_dir="data/xmls/xmls_db"):
    for f in Path(xml_dir).glob("*.xml"):
        try:
            _, measurements, _, _, _ = parse_all(f)
            yield from measurements
        except Exception as e:
            print(f"Failed {f.name}: {e}")

@dlt.resource(name="isolates", write_disposition="replace")
def tce_isolates(xml_dir="data/xmls/xmls_db"):
    for f in Path(xml_dir).glob("*.xml"):
        try:
            _, _, isolates, _, _ = parse_all(f)
            yield from isolates
        except Exception as e:
            print(f"Failed {f.name}: {e}")

@dlt.resource(name="mutations", write_disposition="replace")
def tce_mutations(xml_dir="data/xmls/xmls_db"):
    for f in Path(xml_dir).glob("*.xml"):
        try:
            _, _, _, mutations, _ = parse_all(f)
            yield from mutations
        except Exception as e:
            print(f"Failed {f.name}: {e}")

@dlt.resource(name="treatments", write_disposition="replace")
def tce_treatments(xml_dir="data/xmls/xmls_db"):
    for f in Path(xml_dir).glob("*.xml"):
        try:
            _, _, _, _, treatments = parse_all(f)
            yield from treatments
        except Exception as e:
            print(f"Failed {f.name}: {e}")


@dlt.source
def tce_source():
    return [tce_cases(), tce_measurements(), tce_isolates(), tce_mutations(), tce_treatments()]


pipeline = dlt.pipeline(
    pipeline_name="tce_ingestion",
    destination="filesystem",
    dataset_name="test",
)

load_info = pipeline.run(tce_source(), loader_file_format="parquet")
print(load_info)

Pipeline tce_ingestion load step completed in 2.84 seconds
1 load package(s) were loaded to destination filesystem and into dataset test
The filesystem destination used s3://hiv-data-022784797781 location to store data
Load package 1776550855.062073 is LOADED and contains no failed jobs


---

Dividing the `measurements` Parquet into CD4 and RNA partitions.

In [31]:
session = boto3.Session(profile_name='hiv-project')
s3 = session.client('s3')

# read what dlt uploaded
df = pl.read_parquet(
    "s3://hiv-data-022784797781/test/measurements/*.parquet",
    storage_options={"profile": "hiv-project"}
)

# repartition and upload
for measurement_type, subdf in df.partition_by("measurement_type", as_dict=True).items():
    val = measurement_type[0] if isinstance(measurement_type, tuple) else measurement_type
    subdf = subdf.drop(["measurement_type", "_dlt_id", "_dlt_load_id"])
    
    local_path = f"/tmp/measurements_{val}.parquet"
    subdf.write_parquet(local_path)
    
    s3.upload_file(
        local_path,
        "hiv-data-022784797781",
        f"test/partitioned_measurements/measurement_type={val}/data.parquet"
    )
    print(f"Uploaded measurement_type={val}")

Uploaded measurement_type=RNA
Uploaded measurement_type=CD4


---

Initializing the Glue crawler(s).

In [135]:
def create_crawler(name, s3_path, database='hivdb_tce'):
    """Create Glue crawler for partitioned Parquet data."""
    glue = session.client('glue')
    
    sts = session.client('sts')
    account_id = sts.get_caller_identity()['Account']
    
    role_arn = f'arn:aws:iam::{account_id}:role/glue_crawler_role'
    
    glue.create_crawler(
        Name=name,
        Role=role_arn,
        DatabaseName=database,
        Targets={'S3Targets': [{'Path': s3_path}]},
        SchemaChangePolicy={
            'UpdateBehavior': 'UPDATE_IN_DATABASE',
            'DeleteBehavior': 'DEPRECATE_IN_DATABASE'
        }
    )

create_crawler('dlt_measurements_crawler', f's3://{bucket_name}/test/partitioned_measurements/')

In [136]:
glue = session.client('glue')

In [137]:
glue.start_crawler(Name='dlt_measurements_crawler')

{'ResponseMetadata': {'RequestId': '925595f6-78b1-4e9d-a892-0c11a234159e',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Sat, 18 Apr 2026 23:17:29 GMT',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '2',
   'connection': 'keep-alive',
   'x-amzn-requestid': '925595f6-78b1-4e9d-a892-0c11a234159e',
   'cache-control': 'no-cache'},
  'RetryAttempts': 0}}

In [168]:
glue.get_crawler(Name='dlt_measurements_crawler')['Crawler']['State']

'READY'

In [169]:
glue = session.client('glue')

table = glue.get_table(
    DatabaseName='hivdb_tce',
    Name='partitioned_measurements'
)

cols = table['Table']['StorageDescriptor']['Columns']
[col['Name'] for col in cols]

['xml_filename',
 'patient_alias',
 'relative_date',
 'timepoint_tag',
 'timepoint_type',
 'value',
 'value_type']

---

Athena query checking.

In [170]:
athena = boto3.client("athena", region_name="us-east-1")

In [171]:
def run_athena_query(sql, database="hivdb_tce", output="s3://hiv-data-022784797781/athena-results/"):
    resp = athena.start_query_execution(
        QueryString=sql,
        QueryExecutionContext={"Database": database},
        ResultConfiguration={"OutputLocation": output},
    )
    qid = resp["QueryExecutionId"]

    while True:
        status = athena.get_query_execution(QueryExecutionId=qid)
        state = status["QueryExecution"]["Status"]["State"]

        if state == "SUCCEEDED":
            break
        if state in ("FAILED", "CANCELLED"):
            reason = status["QueryExecution"]["Status"].get("StateChangeReason", "Unknown error")
            raise RuntimeError(f"Athena query {state}: {reason}")

        time.sleep(1)

    results = athena.get_query_results(QueryExecutionId=qid)
    rows = results["ResultSet"]["Rows"]

    headers = [c["VarCharValue"] for c in rows[0]["Data"]]
    data = []
    for row in rows[1:]:
        values = [col.get("VarCharValue") for col in row.get("Data", [])]
        data.append(dict(zip(headers, values)))

    return data

In [172]:
rows = run_athena_query("""
    SELECT *
    FROM hivdb_tce.partitioned_measurements
    LIMIT 10
""")

print(rows[:5])

[{'xml_filename': '162.xml', 'patient_alias': '4303', 'relative_date': '-11', 'timepoint_tag': 'baselineCD4', 'timepoint_type': 'baseline', 'value': '100', 'value_type': 'count', 'measurement_type': 'CD4'}, {'xml_filename': '162.xml', 'patient_alias': '4303', 'relative_date': '-265', 'timepoint_tag': 'pastCD4', 'timepoint_type': 'past', 'value': '40', 'value_type': 'count', 'measurement_type': 'CD4'}, {'xml_filename': '162.xml', 'patient_alias': '4303', 'relative_date': '-215', 'timepoint_tag': 'pastCD4', 'timepoint_type': 'past', 'value': '5', 'value_type': 'count', 'measurement_type': 'CD4'}, {'xml_filename': '162.xml', 'patient_alias': '4303', 'relative_date': '24', 'timepoint_tag': 'followupCD4', 'timepoint_type': 'followup', 'value': '76', 'value_type': 'count', 'measurement_type': 'CD4'}, {'xml_filename': '162.xml', 'patient_alias': '4303', 'relative_date': '31', 'timepoint_tag': 'followupCD4', 'timepoint_type': 'followup', 'value': '64', 'value_type': 'count', 'measurement_type'

In [173]:
rows = run_athena_query("""
    SELECT *
    FROM hivdb_tce.tce_measurements
    LIMIT 10
""")

print(rows[:5])

[{'xml_filename': '162.xml', 'patient_alias': '4303', 'relative_date': '-11', 'timepoint_tag': 'baselineCD4', 'timepoint_type': 'baseline', 'value': '100', 'value_type': 'count', 'measurement_type': 'CD4'}, {'xml_filename': '162.xml', 'patient_alias': '4303', 'relative_date': '-265', 'timepoint_tag': 'pastCD4', 'timepoint_type': 'past', 'value': '40', 'value_type': 'count', 'measurement_type': 'CD4'}, {'xml_filename': '162.xml', 'patient_alias': '4303', 'relative_date': '-215', 'timepoint_tag': 'pastCD4', 'timepoint_type': 'past', 'value': '5', 'value_type': 'count', 'measurement_type': 'CD4'}, {'xml_filename': '162.xml', 'patient_alias': '4303', 'relative_date': '-44', 'timepoint_tag': 'pastCD4', 'timepoint_type': 'past', 'value': '164', 'value_type': 'count', 'measurement_type': 'CD4'}, {'xml_filename': '162.xml', 'patient_alias': '4303', 'relative_date': '-22', 'timepoint_tag': 'pastCD4', 'timepoint_type': 'past', 'value': '156', 'value_type': 'count', 'measurement_type': 'CD4'}]


In [174]:
rows = run_athena_query("""
    SELECT measurement_type, COUNT(*) as cnt
    FROM hivdb_tce.partitioned_measurements
    GROUP BY measurement_type
""")
print(rows)

[{'measurement_type': 'CD4', 'cnt': '22134'}, {'measurement_type': 'RNA', 'cnt': '67158'}]


In [175]:
rows = run_athena_query("""
    SELECT measurement_type, COUNT(*) as cnt
    FROM hivdb_tce.tce_measurements
    GROUP BY measurement_type
""")
print(rows)

[{'measurement_type': 'CD4', 'cnt': '22134'}, {'measurement_type': 'RNA', 'cnt': '67158'}]
